# Regional Storage Preparation

This notebook:

1. downloads EIA underground storage data;
2. selects a storage region (Lower 48 or one of the five EIA regions);
3. calculates weekly storage changes;
4. validates the resulting time series;
5. exports the processed dataset for modeling.

In [ ]:
#imports
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from gas_forecast.data.export import save_versioned_parquet
from gas_forecast.data.regions import region_slug
from gas_forecast.data.storage import (
    fetch_weekly_storage_incremental,
    clean_weekly_storage,
    select_region,
    calculate_weekly_storage_change,
    prepare_storage_model_data,
)

In [ ]:
# configuration
REGION = "R48"  # R48, R01, R02, R03, R04, R05
REGION_SLUG = region_slug(REGION)
CACHE_DIR = Path("../data/cache")
PROCESSED_DIR = Path("../data/processed")

load_dotenv("local.env")
API_KEY = os.getenv("EIA_API_KEY")

In [ ]:
# load raw storage data (incremental cache)

storage_raw = fetch_weekly_storage_incremental(
    API_KEY,
    cache_dir=CACHE_DIR,
    revision_weeks=8,
)

storage = clean_weekly_storage(
    storage_raw,
    start_date="2010-01-01",
)

In [ ]:
# select region
region_storage = select_region(storage, REGION)

In [ ]:
# calculate weekly change
region_storage = calculate_weekly_storage_change(region_storage)

In [ ]:
region_storage.head()

In [ ]:
# export to parquet
region_weekly_storage = prepare_storage_model_data(region_storage)

save_versioned_parquet(
    region_weekly_storage,
    PROCESSED_DIR,
    f"{REGION_SLUG}_weekly_storage",
    save_latest=True,
)